# Evidence Aggregation Ablation
## RAGTruth $\rightarrow$ PublicHearingBR

This read-only notebook documents the frozen evidence-aggregation comparison across four RAGTruth confirmatory campaigns: the original pooling ablation (Gated Attention, Mean, and Max) plus the compositional Set Transformer extension. The original methods aggregate `projected [B,4,128]` directly; Set Transformer models interactions between the four chunks before PMA aggregation.

**Decision criterion: PublicHearingBR zero-shot AUPRC.** The target is imbalanced (501/4,235 positives), so AUPRC is the primary ranking criterion. AUROC and Brier are complementary discrimination and score-quality diagnostics.

## Reproducible configuration

Configuration precedence is `NOTEBOOK_CONFIG`, then `notebooks/configs/pooling_ablation.local.yaml`. Copy `pooling_ablation.example.yaml` to that local path to select different frozen campaign directories. Paths must remain repository-relative. This notebook reads existing runs and bootstrap outputs only; it does not load models or modify artifacts.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display


POOLINGS = ("gated_attention", "mean", "max", "set_transformer")
LABELS = {
    "gated_attention": "Gated Attention",
    "mean": "Mean",
    "max": "Max",
    "set_transformer": "Set Transformer",
}
COLORS = {
    "gated_attention": "#1f77b4",
    "mean": "#f4a261",
    "max": "#2a9d8f",
    "set_transformer": "#8e5ea2",
}


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "notebooks" / "configs" / "pooling_ablation.example.yaml").is_file():
            return candidate
    raise FileNotFoundError("Repository root with the pooling ablation configuration was not found.")


def resolve_repository_path(root: Path, value: object, label: str) -> Path:
    relative_path = Path(value) if isinstance(value, str) else None
    if not relative_path or relative_path.is_absolute() or ".." in relative_path.parts:
        raise ValueError(f"{label} must define a repository-relative path.")
    return root / relative_path


def load_notebook_config(root: Path) -> tuple[Path, dict[str, Path], dict[str, Path]]:
    requested = os.environ.get("NOTEBOOK_CONFIG")
    config_path = Path(requested) if requested else root / "notebooks" / "configs" / "pooling_ablation.local.yaml"
    if not config_path.is_absolute():
        config_path = root / config_path
    if not config_path.is_file():
        raise FileNotFoundError(
            "Local notebook configuration not found. Copy\n"
            "notebooks/configs/pooling_ablation.example.yaml\n"
            "to notebooks/configs/pooling_ablation.local.yaml and configure the campaign paths."
        )
    raw = yaml.safe_load(config_path.read_text(encoding="utf-8"))
    experiments = raw.get("experiments") if isinstance(raw, dict) else None
    if not isinstance(experiments, dict):
        raise ValueError("Notebook configuration requires an experiments mapping.")
    resolved: dict[str, Path] = {}
    for pooling in POOLINGS:
        entry = experiments.get(pooling)
        value = entry.get("path") if isinstance(entry, dict) else None
        resolved[pooling] = resolve_repository_path(root, value, pooling)
    analysis = raw.get("analysis")
    if not isinstance(analysis, dict):
        raise ValueError("Notebook configuration requires an analysis mapping.")
    bootstrap_paths: dict[str, Path] = {}
    for name in ("pooling_paired_bootstrap", "set_transformer_paired_bootstrap"):
        entry = analysis.get(name)
        value = entry.get("path") if isinstance(entry, dict) else None
        bootstrap_paths[name] = resolve_repository_path(root, value, name)
    return config_path, resolved, bootstrap_paths


def load_json(path: Path) -> dict:
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)


def add_bar_labels(axis, bars, errors=None, fmt="{:.3f}", padding=0.012) -> None:
    errors = np.zeros(len(bars)) if errors is None else np.asarray(errors)
    for bar, error in zip(bars, errors):
        axis.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + error + padding, fmt.format(bar.get_height()), ha="center", va="bottom", fontsize=8)


ROOT = find_repo_root()
os.chdir(ROOT)
config_path, experiment_paths, bootstrap_paths = load_notebook_config(ROOT)
config_label = config_path.relative_to(ROOT) if config_path.is_relative_to(ROOT) else config_path
print(f"Configuration: {config_label}")
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

## Frozen protocol parity

Mean, Max, Gated Attention, and Set Transformer use the same frozen dataset, grouped split, seeds, encoder, LoRA, optimization, and classifier settings. The original pooling comparison is Gated Attention versus Mean/Max; Set Transformer is an extension with its explicit SAB/PMA aggregator settings. This cell verifies equality of all non-aggregator configuration fields.

In [ ]:
campaigns = {}
per_seed_frames = []
protocol_rows = []

for pooling in POOLINGS:
    run_dir = experiment_paths[pooling]
    if not run_dir.is_dir():
        raise FileNotFoundError(f"Campaign directory not found: {run_dir.relative_to(ROOT)}")
    frozen = load_json(run_dir / "frozen_protocol.json")
    root_manifest = load_json(run_dir / "manifest.json")
    payload = frozen["payload"]
    experiment = payload["experiment_config"]
    campaigns[pooling] = {"dir": run_dir, "payload": payload, "experiment": experiment}
    protocol_rows.append({
        "aggregator": LABELS[pooling],
        "run signature": frozen["signature"],
        "architecture": experiment["architecture"],
        "dataset signature": payload["dataset_signature"],
        "split signature": payload["split_signature"],
        "seeds": str(payload["model_seeds"]),
        "selection": f"{payload['selection_metric']} ({payload['selection_mode']})",
        "status": root_manifest["status"],
    })
    frame = pd.read_csv(run_dir / "aggregate" / "per_seed_metrics.csv")
    frame["pooling"] = pooling
    per_seed_frames.append(frame)

reference = campaigns["gated_attention"]["experiment"].copy()
for field in ("run_name", "architecture", "pooling_type", "set_transformer"):
    reference.pop(field, None)
for pooling in ("mean", "max", "set_transformer"):
    candidate = campaigns[pooling]["experiment"].copy()
    for field in ("run_name", "architecture", "pooling_type", "set_transformer"):
        candidate.pop(field, None)
    assert candidate == reference, f"Non-aggregator configuration differs for {pooling}."

metrics = pd.concat(per_seed_frames, ignore_index=True)
display(pd.DataFrame(protocol_rows))
print("Protocol check passed: frozen configurations are identical outside aggregator identity and Set Transformer settings.")

## Threshold-free ranking performance

Continuous-score metrics are shown as mean $\pm$ standard deviation across seeds. They do not depend on a decision threshold. AUPRC is primary for the imbalanced target; AUROC and Brier provide complementary views.

In [ ]:
threshold_free = metrics.loc[metrics["threshold_regime"].eq("threshold_free")].copy()
summary = threshold_free.groupby(["dataset", "pooling"])[["AUPRC", "AUROC", "Brier"]].agg(["mean", "std"])

figure, axes = plt.subplots(1, 2, figsize=(12.0, 4.0), sharey=True)
for axis, dataset, title in zip(axes, ("ragtruth_test", "publichearing_zero_shot"), ("RAGTruth test", "PublicHearingBR zero-shot")):
    x = np.arange(2)
    width = 0.18
    for index, pooling in enumerate(POOLINGS):
        values = [summary.loc[(dataset, pooling), (metric, "mean")] for metric in ("AUPRC", "AUROC")]
        errors = [summary.loc[(dataset, pooling), (metric, "std")] for metric in ("AUPRC", "AUROC")]
        bars = axis.bar(x + (index - 1.5) * width, values, width, yerr=errors, capsize=3, label=LABELS[pooling], color=COLORS[pooling])
        add_bar_labels(axis, bars, errors)
    axis.set_xticks(x, ("AUPRC", "AUROC"))
    axis.set_ylim(0, 1)
    axis.set_title(title, loc="left", weight="bold")
    axis.grid(axis="y", alpha=0.2)
axes[0].set_ylabel("Score")
handles, labels = axes[0].get_legend_handles_labels()
figure.legend(handles, labels, frameon=False, fontsize=8, loc="center left", bbox_to_anchor=(0.86, 0.5))
figure.suptitle("Threshold-free ranking performance", x=0.08, ha="left", weight="bold")
figure.tight_layout(rect=(0, 0, 0.83, 1))
plt.show()

display(summary.rename(index=LABELS, level="pooling").round(4))

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(9.5, 4.2), sharey=True)
for axis, dataset, title in zip(axes, ("ragtruth_test", "publichearing_zero_shot"), ("RAGTruth test", "PublicHearingBR zero-shot")):
    values = [summary.loc[(dataset, pooling), ("Brier", "mean")] for pooling in POOLINGS]
    errors = [summary.loc[(dataset, pooling), ("Brier", "std")] for pooling in POOLINGS]
    bars = axis.bar([LABELS[pooling] for pooling in POOLINGS], values, yerr=errors, capsize=3, color=[COLORS[pooling] for pooling in POOLINGS])
    add_bar_labels(axis, bars, errors)
    axis.set_ylim(0, max(np.asarray(values) + np.asarray(errors)) + 0.060)
    axis.set_title(title, loc="left", weight="bold", pad=14)
    axis.grid(axis="y", alpha=0.2)
    axis.tick_params(axis="x", rotation=18)
axes[0].set_ylabel("Brier score (lower is better)")
figure.suptitle("Score quality diagnostic", x=0.08, ha="left", weight="bold")
figure.tight_layout()
plt.show()

## Frozen threshold transfer

Each seed selects `best_f1` and `fpr10` on RAGTruth validation only. The selected threshold is then frozen and applied to RAGTruth test and PublicHearingBR. The target labels do not participate in threshold selection. Set Transformer is included because equivalent thresholded artifacts are available.

In [ ]:
target_operating = metrics.loc[
    metrics["dataset"].eq("publichearing_zero_shot") & metrics["threshold_regime"].isin(("best_f1", "fpr10"))
].copy()
operating_summary = target_operating.groupby(["threshold_regime", "pooling"])[["F1", "Recall", "FPR"]].agg(["mean", "std"])

figure, axes = plt.subplots(1, 2, figsize=(12.0, 3.8))
for axis, metric, title in zip(axes, ("F1", "FPR"), ("F1 (higher is better)", "False-positive rate (lower is better)")):
    x = np.arange(2)
    width = 0.18
    for index, pooling in enumerate(POOLINGS):
        values = [operating_summary.loc[(regime, pooling), (metric, "mean")] for regime in ("best_f1", "fpr10")]
        errors = [operating_summary.loc[(regime, pooling), (metric, "std")] for regime in ("best_f1", "fpr10")]
        bars = axis.bar(x + (index - 1.5) * width, values, width, yerr=errors, capsize=3, label=LABELS[pooling], color=COLORS[pooling])
        add_bar_labels(axis, bars, errors)
    axis.set_xticks(x, ("best F1", "FPR 10%"))
    axis.set_ylim(0, 0.7 if metric == "F1" else 0.13)
    axis.set_title(title, loc="left", weight="bold")
    axis.grid(axis="y", alpha=0.2)
axes[0].set_ylabel("Score")
handles, labels = axes[0].get_legend_handles_labels()
figure.legend(handles, labels, frameon=False, fontsize=8, loc="center left", bbox_to_anchor=(0.86, 0.5))
figure.suptitle("PublicHearingBR operating points from frozen RAGTruth thresholds", x=0.08, ha="left", weight="bold")
figure.tight_layout(rect=(0, 0, 0.83, 1))
plt.show()

display(operating_summary.rename(index=LABELS, level="pooling").round(4))

## Paired grouped bootstrap

The original pooling bootstrap compares Gated Attention with Mean and Max using the same 4,235 `example_id`s and 206 `hearing_id`s. It resamples hearings with replacement, preserves group multiplicity, uses the same sampled hearings across methods and seeds, and averages the three seed effects. A separate completed bootstrap compares Set Transformer with historical Gated Attention under the same semantics.

Set Transformer is a compositional extension: Mean is uniform aggregation, Max emphasizes extreme signals, and Gated Attention learns evidence importance. Set Transformer applies one self-attention block before PMA, allowing chunks to interact before aggregation. With Top-4 retrieval, complementary evidence can occur in different chunks; this is a plausible architectural interpretation, not a causal claim.

In [ ]:
pooling_bootstrap_summary = load_json(bootstrap_paths["pooling_paired_bootstrap"] / "comparison_summary.json")
set_bootstrap_summary = load_json(bootstrap_paths["set_transformer_paired_bootstrap"] / "comparison_summary.json")

bootstrap_rows = []
for comparison_name, comparison in pooling_bootstrap_summary.items():
    left, right = comparison["left"], comparison["right"]
    observed = comparison["observed"]
    for metric in ("auprc", "auroc", "brier"):
        result = comparison["campaign"][metric]
        delta = observed["delta_mean_across_seeds"][metric]
        if result["ci_crosses_zero"]:
            conclusion = "Inconclusive"
        elif metric == "brier":
            conclusion = f"{LABELS[left if delta < 0 else right]} lower Brier"
        else:
            conclusion = f"{LABELS[left if delta > 0 else right]} higher {metric.upper()}"
        bootstrap_rows.append({
            "Comparison": f"{LABELS[left]} vs {LABELS[right]}",
            "Metric": metric.upper(),
            "Left point": observed["left_mean_across_seeds"][metric],
            "Right point": observed["right_mean_across_seeds"][metric],
            "Delta": delta,
            "95% CI": f"[{result['ci_lower']:+.4f}, {result['ci_upper']:+.4f}]",
            "Crosses zero": result["ci_crosses_zero"],
            "Conclusion": conclusion,
        })

set_comparison = set_bootstrap_summary
set_observed = set_comparison["observed"]
for metric in ("auprc", "auroc", "brier"):
    result = set_comparison["campaign"][metric]
    delta = set_observed["delta_set_minus_gated"][metric]
    conclusion = "Inconclusive" if result["ci_crosses_zero"] else ("Set Transformer lower Brier" if metric == "brier" and delta < 0 else "Set Transformer higher " + metric.upper())
    bootstrap_rows.append({
        "Comparison": "Set Transformer vs Gated Attention",
        "Metric": metric.upper(),
        "Left point": set_observed["set_transformer"][metric],
        "Right point": set_observed["gated_attention"][metric],
        "Delta": delta,
        "95% CI": f"[{result['ci_lower']:+.4f}, {result['ci_upper']:+.4f}]",
        "Crosses zero": result["ci_crosses_zero"],
        "Conclusion": conclusion,
    })

bootstrap_table = pd.DataFrame(bootstrap_rows)
display(bootstrap_table)

attention_max_auprc = pooling_bootstrap_summary["gated_attention_vs_max"]["campaign"]["auprc"]
attention_mean_auprc = pooling_bootstrap_summary["gated_attention_vs_mean"]["campaign"]["auprc"]
print(f"Gated vs Max AUPRC: IC95% [{attention_max_auprc['ci_lower']:+.4f}, {attention_max_auprc['ci_upper']:+.4f}]; crosses zero = {attention_max_auprc['ci_crosses_zero']}.")
print(f"Gated vs Mean AUPRC: IC95% [{attention_mean_auprc['ci_lower']:+.4f}, {attention_mean_auprc['ci_upper']:+.4f}]; crosses zero = {attention_mean_auprc['ci_crosses_zero']}.")
set_auprc = set_comparison["campaign"]["auprc"]
print(f"Set vs Gated AUPRC: IC95% [{set_auprc['ci_lower']:+.4f}, {set_auprc['ci_upper']:+.4f}]; crosses zero = {set_auprc['ci_crosses_zero']}.")

In [ ]:
comparison_sources = {
    **{name: comparison for name, comparison in pooling_bootstrap_summary.items()},
    "set_transformer_vs_gated_attention": set_comparison,
}
comparison_order = ("gated_attention_vs_max", "gated_attention_vs_mean", "mean_vs_max", "set_transformer_vs_gated_attention")
metric_titles = {
    "auprc": "AUPRC (higher is better)",
    "auroc": "AUROC (higher is better)",
    "brier": "Brier score (lower is better)",
}


def observed_delta(comparison, metric):
    observed = comparison["observed"]
    if "delta_set_minus_gated" in observed:
        return observed["delta_set_minus_gated"][metric]
    return observed["delta_mean_across_seeds"][metric]


def comparison_label(name):
    comparison = comparison_sources[name]
    if name == "set_transformer_vs_gated_attention":
        return "Set Transformer vs Gated Attention"
    return f"{LABELS[comparison['left']]} vs {LABELS[comparison['right']]}"


figure, axes = plt.subplots(1, 3, figsize=(12.5, 4.8), sharey=True)
y = np.arange(len(comparison_order))
comparison_labels = [comparison_label(name) for name in comparison_order]

for axis, metric in zip(axes, ("auprc", "auroc", "brier")):
    points, lower, upper, colors = [], [], [], []
    for name in comparison_order:
        comparison = comparison_sources[name]
        result = comparison["campaign"][metric]
        point = observed_delta(comparison, metric)
        points.append(point)
        lower.append(point - result["ci_lower"])
        upper.append(result["ci_upper"] - point)
        colors.append("#6c757d" if result["ci_crosses_zero"] else "#1f77b4")
    axis.axvline(0, color="black", linewidth=1, zorder=0)
    for position, point, low, high, color in zip(y, points, lower, upper, colors):
        axis.errorbar(point, position, xerr=[[low], [high]], fmt="o", color=color, capsize=3, markersize=5)
    axis.set_title(metric_titles[metric], loc="left", weight="bold")
    axis.grid(axis="x", alpha=0.2)
    axis.set_xlabel("Delta")
    axis.set_yticks(y, comparison_labels)
    axis.invert_yaxis()

figure.suptitle("PublicHearingBR zero-shot: paired grouped bootstrap effects", x=0.08, ha="left", weight="bold")
figure.text(0.08, 0.01, "Blue: IC95% excludes zero; gray: inconclusive. For Brier, negative favors the left model.", fontsize=8)
figure.tight_layout(rect=(0, 0.06, 1, 1))
plt.show()

## Final scientific conclusion

The original pooling bootstrap supports retaining Gated Attention over Mean and Max on the pre-specified primary criterion: its AUPRC deltas are positive and their 95% intervals exclude zero. Set Transformer has the best descriptive mean across AUPRC, AUROC, and Brier, but its AUPRC advantage over Gated Attention has a 95% interval that crosses zero. Therefore, the Set Transformer AUPRC improvement is not conclusively established under the paired grouped bootstrap.

The Set Transformer comparison does provide evidence favorable to Set Transformer for AUROC and Brier. These secondary results do not replace the pre-specified AUPRC criterion, so there is insufficient evidence to replace Gated Attention on that criterion. Set Transformer remains a strong compositional variant, without motivating a new architectural tuning round in this analysis.

In [ ]:
target_ranking = (
    threshold_free.loc[threshold_free["dataset"].eq("publichearing_zero_shot")]
    .groupby("pooling")[["AUPRC", "AUROC", "Brier"]]
    .agg(["mean", "std"])
    .reindex(POOLINGS)
)
display(target_ranking.rename(index=LABELS).round(4))

for metric, direction in (("AUPRC", "idxmax"), ("AUROC", "idxmax"), ("Brier", "idxmin")):
    winner = getattr(target_ranking[(metric, "mean")], direction)()
    print(f"Descriptive PublicHearingBR zero-shot {metric} winner: {LABELS[winner]} ({target_ranking.loc[winner, (metric, 'mean')]:.4f}).")

set_auprc = set_comparison["campaign"]["auprc"]
print("Primary criterion: AUPRC.")
print(f"Set Transformer vs Gated Attention AUPRC IC95% crosses zero: {set_auprc['ci_crosses_zero']}. Conclusion remains descriptive/inconclusive.")